In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="boniromou/zh-yue-tts-dataset", 
    repo_type="dataset", local_dir="./zh-yue-tts-dataset", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 3 files: 100%|██████████| 3/3 [00:04<00:00,  1.66s/it]


'/home/ubuntu/zh-yue-tts-dataset'

In [3]:
files = glob('zh-yue-tts-dataset/*/*.parquet')
len(files)

3

In [4]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [5]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 12714/12714 [33:38<00:00,  6.30it/s]


In [6]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'zh-yue-tts-dataset_audio/zh-yue-tts-dataset-data-train-00000-of-00003_0.mp3',
 'text': '這個問題已經陷入咗死衚衕，我哋需要猜出新嘅解決方法。',
 'speaker': 'zh-yue-tts-dataset_audio'}

In [7]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'zh-yue-tts-dataset')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 58.64ba/s]
Processing Files (1 / 1): 100%|██████████| 2.44MB / 2.44MB,  0.00B/s  
New Data Upload: 100%|██████████| 2.44MB / 2.44MB,  0.00B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.58 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/c5731a193ffa14b9702cb01c41dab9225410ae99', commit_message='Upload dataset', commit_description='', oid='c5731a193ffa14b9702cb01c41dab9225410ae99', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [8]:
audio_files = [d['audio_filename'] for d in data]

with open('zh-yue-tts-dataset-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [10]:
# !zip -rq zh-yue-tts-dataset.zip zh-yue-tts-dataset
# !hf upload malaysia-ai/Multilingual-TTS zh-yue-tts-dataset.zip --repo-type=dataset